# NinaPro DB2 exploratory analysis for sEMG diffusion research

This notebook examines one real NinaPro DB2 recording before any model is trained. The purpose is to understand what information exists in the raw multichannel time series and what a synthetic generator will eventually need to preserve.

**Evidence status:** every waveform and numerical result below is measured from Subject 1, Exercise 1. This is exploratory evidence, not a population-level result.

## Learning objectives

By the end of the notebook, I should be able to explain:

1. why sEMG is treated as a multichannel time series;
2. how refined gesture and repetition labels define a trial;
3. what RMS amplitude says about muscle activation across electrodes;
4. what channel correlation reveals about spatial structure;
5. how the power spectrum and spectrogram describe frequency content;
6. how an RMS envelope differs from the Teager--Kaiser energy operator (TKEO); and
7. why repetition variability matters for trustworthy synthetic generation.

In [ ]:
import importlib
import os
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch

working_directory = Path.cwd().resolve()
override = os.environ.get("EMG_DIFFUSION_ROOT")
repo_candidates = [
    working_directory,
    *working_directory.parents,
    Path("/home/nvidia/Ayorinde_Workspace/EMG_Diffusion"),
    Path.home() / "research" / "DDPM-EMG-Research",
]
if override:
    repo_candidates.insert(0, Path(override).expanduser().resolve())
REPO_ROOT = next(
    (path for path in repo_candidates if (path / "src" / "emg_diffusion").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root. Set EMG_DIFFUSION_ROOT to the "
        "absolute path of the cloned repository and rerun this cell. "
        f"Current working directory: {working_directory}"
    )
sys.path.insert(0, str(REPO_ROOT / "src"))

import emg_diffusion.visualization.eda as eda_module
importlib.reload(eda_module)

from emg_diffusion.visualization.eda import (
    gesture_channel_rms,
    load_ninapro_record,
    movement_rms_profile,
    plot_activation_dynamics,
    plot_channel_structure,
    plot_gesture_channel_atlas,
    plot_record_overview,
    plot_repetition_variability,
    plot_rest_active_rms,
    plot_spectral_view,
    plot_subject_channel_profiles,
    plot_trial_channels,
    plot_trial_pca,
    principal_component_projection,
    save_figure,
    select_trial,
    state_window_rms,
    trial_feature_matrix,
)

plt.rcParams.update({
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.facecolor": "white",
    "svg.fonttype": "none",
})

FIGURE_DIR = REPO_ROOT / "outputs" / "figures" / "eda"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
print(f"Repository: {REPO_ROOT}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Visible GPUs: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"GPU 0: {torch.cuda.get_device_name(0)}")
print("The EDA uses CPU calculations; GPUs will matter during model training.")

## 1. Load one audited recording

The first analysis uses `S1_E1_A1.mat`: Subject 1, Exercise 1, acquisition 1. We use the refined `restimulus` and `rerepetition` arrays because they provide the corrected gesture boundaries used by the audit.

In [ ]:
DATA_ROOT = REPO_ROOT / "data" / "raw" / "ninapro_db2"
matches = list(DATA_ROOT.rglob("S1_E1_A1.mat"))
if len(matches) != 1:
    raise FileNotFoundError(
        f"Expected one S1_E1_A1.mat below {DATA_ROOT}; found {len(matches)}"
    )

record = load_ninapro_record(matches[0], label_source="refined")
active_labels = np.unique(record.labels[record.labels > 0])
active_repetitions = np.unique(record.repetitions[record.repetitions > 0])

print(f"File: {record.path.name}")
print(f"EMG shape: {record.emg.shape[0]:,} samples x {record.emg.shape[1]} channels")
print(f"Sampling rate: {record.sampling_rate_hz:,} Hz")
print(f"Duration: {record.duration_seconds / 60:.2f} minutes")
print(f"Active gesture labels: {active_labels.tolist()}")
print(f"Repetitions: {active_repetitions.tolist()}")
print(f"All EMG values finite: {np.isfinite(record.emg).all()}")
print(f"Terminal samples removed during alignment: {record.removed_samples}")

## 2. Look at the multichannel signal and labels

The traces are independently scaled by their 99th-percentile absolute amplitude and vertically offset. This makes timing and burst structure visible, but it must **not** be used to compare absolute channel amplitudes. Gesture label 0 means rest.

**What to look for:** bursts should appear at related times across several channels, while their shapes and strengths differ because the electrodes observe different muscle regions.

In [ ]:
figure = plot_record_overview(record, start_seconds=0, duration_seconds=12)
overview_paths = save_figure(
    figure,
    FIGURE_DIR,
    "01_multichannel_overview",
    title="NinaPro DB2 Subject 1 multichannel sEMG overview",
    description=("Twelve independently scaled sEMG channels above the refined "
                 "gesture label over the first twelve seconds."),
)
plt.show()
print("Saved:", *overview_paths, sep="\n- ")

## 3. Isolate one labelled movement trial

A trial is a contiguous interval for one gesture and one repetition. We start with gesture 1, repetition 1. Keeping trial boundaries intact is important: randomly cutting overlapping windows before subject and repetition splitting can cause leakage.

**What to look for:** channels do not carry identical waveforms. The generator must preserve their shared timing without erasing channel-specific structure.

In [ ]:
MOVEMENT = 1
REPETITION = 1
trial = select_trial(record, movement=MOVEMENT, repetition=REPETITION)
trial_duration = trial.sample_count / record.sampling_rate_hz
print(
    f"Selected samples [{trial.start_sample:,}, {trial.stop_sample:,}) "
    f"with duration {trial_duration:.3f} s"
)

figure = plot_trial_channels(record, trial)
trial_paths = save_figure(
    figure, FIGURE_DIR, "02_selected_trial_channels",
    title="Gesture 1 repetition 1 across twelve sEMG channels",
    description=("Vertically offset channel traces show shared activation timing "
                 "and channel-specific waveform structure."),
)
plt.show()

## 4. Measure spatial structure across electrodes

RMS is the square root of mean squared amplitude and summarizes activation strength over the selected trial. The correlation matrix measures zero-lag linear similarity between pairs of channels.

These measurements answer different questions: RMS asks **how strongly** a channel is active, while correlation asks **how similarly** two channels vary over time. A diffusion model that generates each channel independently would struggle to reproduce this joint structure.

In [ ]:
trial_signal = record.emg[trial.start_sample:trial.stop_sample]
channel_rms = np.sqrt(np.mean(np.square(trial_signal), axis=0))
MOST_ACTIVE_CHANNEL = int(np.argmax(channel_rms)) + 1
print(f"Highest-RMS channel in this trial: {MOST_ACTIVE_CHANNEL}")
print(f"RMS range across channels: {channel_rms.min():.6g} to {channel_rms.max():.6g}")

figure = plot_channel_structure(record, trial)
spatial_paths = save_figure(
    figure, FIGURE_DIR, "03_channel_structure",
    title="Amplitude and correlation structure across sEMG channels",
    description=("Channel RMS amplitudes are shown beside the twelve-by-twelve "
                 "Pearson correlation matrix for one gesture trial."),
)
plt.show()

## 5. Examine frequency and time--frequency content

The Welch power spectral density estimates how signal power is distributed over frequency. Mean and median frequency summarize that distribution. The spectrogram adds time, revealing whether the frequency content changes within the movement.

A spectrum averages over the entire trial; a spectrogram preserves temporal changes but trades frequency resolution for time resolution. Neither replaces the raw waveform.

In [ ]:
figure, spectral_features = plot_spectral_view(
    record, trial, channel=MOST_ACTIVE_CHANNEL
)
spectral_paths = save_figure(
    figure, FIGURE_DIR, "04_spectral_view",
    title="Frequency and time-frequency views of one sEMG channel",
    description=("A Welch power spectrum with mean and median frequency markers "
                 "is paired with a spectrogram for the same trial."),
)
plt.show()
for name, value in spectral_features.items():
    print(f"{name}: {value:.6e}" if name == "total_power" else f"{name}: {value:.4f}")

## 6. Compare raw amplitude, RMS envelope, and TKEO

The raw signal contains rapid positive and negative fluctuations. A moving RMS envelope summarizes local signal magnitude and makes slower activation timing easier to see. TKEO uses

$$\Psi[x_n] = x_n^2 - x_{n-1}x_{n+1},$$

so it responds to local changes in both amplitude and frequency. It is an energy-sensitive nonlinear feature, not a denoised version of the signal. The three curves are robustly scaled for shape comparison and therefore do not share physical units.

In [ ]:
figure = plot_activation_dynamics(
    record,
    trial,
    channel=MOST_ACTIVE_CHANNEL,
    envelope_window_ms=50,
)
activation_paths = save_figure(
    figure, FIGURE_DIR, "05_activation_dynamics",
    title="Raw sEMG, RMS envelope, and TKEO activation views",
    description=("Three aligned traces compare rapid raw fluctuations, the "
                 "fifty-millisecond RMS envelope, and local TKEO energy."),
)
plt.show()

## 7. Examine natural variation across repetitions

The heatmap compares RMS values for the same gesture across its six repetitions. Each value is divided by that channel's median across repetitions, so 1.0 means typical activation for that electrode.

A trustworthy generator should not reproduce one average-looking signal repeatedly. It should represent plausible repetition-to-repetition variation while retaining gesture identity and channel relationships.

In [ ]:
figure, repetition_ids, repetition_rms = plot_repetition_variability(
    record, movement=MOVEMENT
)
repetition_paths = save_figure(
    figure, FIGURE_DIR, "06_repetition_variability",
    title="Gesture 1 RMS variability across repetitions and channels",
    description=("A six-by-twelve heatmap shows repetition RMS relative to each "
                 "channel's median across repetitions."),
)
plt.show()

coefficient_of_variation = repetition_rms.std(axis=0) / repetition_rms.mean(axis=0)
print("Channel-wise RMS coefficient of variation:")
for channel, value in enumerate(coefficient_of_variation, start=1):
    print(f"  Channel {channel:02d}: {value:.3f}")

## 8. Compare rest and active windows across every channel

We now move beyond one trial. The full Exercise 1 recording is divided into non-overlapping 200 ms windows. A window is called **rest** only when every label is 0, and **active** only when every label is the same non-zero gesture. Windows containing a label transition are excluded.

The logarithmic boxplots show distributions rather than one average. Separation between rest and active RMS indicates amplitude information that may help recognition, while overlap reminds us that amplitude alone cannot define a gesture.

In [ ]:
WINDOW_MS = 200
rest_rms, active_rms, excluded_windows = state_window_rms(
    record, window_ms=WINDOW_MS, step_ms=WINDOW_MS
)
median_ratio = np.median(active_rms, axis=0) / np.median(rest_rms, axis=0)
print(f"Rest-only windows: {rest_rms.shape[0]:,}")
print(f"Single-gesture windows: {active_rms.shape[0]:,}")
print(f"Transition windows excluded: {excluded_windows:,}")
print(
    "Active/rest median RMS ratio across channels: "
    f"{median_ratio.min():.2f} to {median_ratio.max():.2f}"
)

figure = plot_rest_active_rms(rest_rms, active_rms, window_ms=WINDOW_MS)
rest_active_paths = save_figure(
    figure, FIGURE_DIR, "07_rest_active_rms",
    title="Rest and active RMS distributions across twelve sEMG channels",
    description=("Paired channel-wise boxplots compare transition-free rest and "
                 "single-gesture windows using a logarithmic amplitude axis."),
)
plt.show()

## 9. Build a gesture-by-channel activation atlas

For every one of the 17 gestures, RMS is calculated for each of its six repetitions and averaged by channel. Panel (a) preserves absolute differences in recorded amplitude. Panel (b) divides each gesture by its strongest channel, revealing its relative spatial recruitment profile.

A conditional generator should reproduce both properties: plausible absolute strength and the gesture-dependent pattern across electrodes.

In [ ]:
movements, gesture_mean_rms, gesture_rms_std = gesture_channel_rms(record)
peak_channel_by_gesture = np.argmax(gesture_mean_rms, axis=1) + 1
print(f"Gesture-channel matrix: {gesture_mean_rms.shape}")
print("Peak-RMS channel by gesture:")
print(dict(zip(movements.tolist(), peak_channel_by_gesture.tolist())))

figure = plot_gesture_channel_atlas(movements, gesture_mean_rms)
gesture_atlas_paths = save_figure(
    figure, FIGURE_DIR, "08_gesture_channel_atlas",
    title="Gesture-by-channel RMS activation atlas for Subject 1",
    description=("Two heatmaps show absolute mean RMS and the within-gesture "
                 "channel profile for seventeen gestures and twelve channels."),
)
plt.show()

## 10. Ask whether simple trial features contain gesture structure

Each of the 102 trials is represented by 24 features: log RMS and log mean waveform length for all 12 channels. PCA standardizes these features and finds directions of greatest overall variation without using the gesture labels.

Each point is one repetition, and the numbered circles mark gesture centroids. Separation suggests that simple amplitude and waveform-change features contain gesture information. Overlap does **not** prove that gestures are inseparable: PCA is linear, unsupervised, and only two components are displayed.

In [ ]:
trial_features, trial_movements, trial_repetitions, feature_names = (
    trial_feature_matrix(record)
)
pca_scores, explained_ratio, pca_components = principal_component_projection(
    trial_features
)
print(f"Feature matrix: {trial_features.shape[0]} trials x {trial_features.shape[1]} features")
print(f"Variance displayed by PC1 and PC2: {100 * explained_ratio[:2].sum():.2f}%")

figure = plot_trial_pca(pca_scores, explained_ratio, trial_movements)
pca_paths = save_figure(
    figure, FIGURE_DIR, "09_trial_feature_pca",
    title="PCA of trial-level multichannel sEMG features",
    description=("One hundred and two trials are projected using RMS and mean "
                 "waveform-length features; numbered markers identify gesture centroids."),
)
plt.show()

## 11. Examine cross-subject variation

The final exploratory comparison uses the fixed subject subset `[1, 10, 20, 30, 40]`. For Gesture 1, each subject's channel RMS is averaged over six repetitions. This subset is chosen for computationally light visualization and is not a statistical sample or final evaluation cohort.

Panel (a) shows absolute amplitude differences. Panel (b) removes each subject's overall scale and emphasizes the relative channel pattern. Variation in either panel is a direct motivation for subject conditioning and subject-disjoint evaluation.

In [ ]:
SUBJECT_IDS = np.array([1, 10, 20, 30, 40])
subject_profiles = []
for subject_id in SUBJECT_IDS:
    if subject_id == record.subject:
        subject_record = record
    else:
        subject_matches = list(DATA_ROOT.rglob(f"S{subject_id}_E1_A1.mat"))
        if len(subject_matches) != 1:
            raise FileNotFoundError(
                f"Expected one Exercise 1 file for Subject {subject_id}"
            )
        subject_record = load_ninapro_record(
            subject_matches[0], label_source="refined"
        )
    subject_profiles.append(movement_rms_profile(subject_record, MOVEMENT))

subject_profiles = np.vstack(subject_profiles)
peak_channels = np.argmax(subject_profiles, axis=1) + 1
print("Peak-RMS channel for Gesture 1 in the fixed subset:")
print(dict(zip(SUBJECT_IDS.tolist(), peak_channels.tolist())))

figure = plot_subject_channel_profiles(
    SUBJECT_IDS, subject_profiles, movement=MOVEMENT
)
subject_paths = save_figure(
    figure, FIGURE_DIR, "10_cross_subject_channel_profiles",
    title="Cross-subject variation in Gesture 1 channel profiles",
    description=("Absolute and within-subject normalized RMS heatmaps compare "
                 "Gesture 1 across five fixed NinaPro DB2 subjects."),
)
plt.show()

## 12. What this means for the diffusion study

This single-record exploration motivates several measurable requirements for the eventual generator:

- **Temporal fidelity:** reproduce burst timing, envelopes, and within-trial dynamics.
- **Spectral fidelity:** reproduce power distributions without treating a spectral image as the signal itself.
- **Spatial fidelity:** preserve channel-specific amplitudes and cross-channel dependence.
- **Conditional correctness:** generated signals must agree with the requested gesture and available subject context.
- **Diversity:** represent realistic repetition-level variation without memorizing training trials.
- **Cross-subject generalization:** handle changes in absolute amplitude and spatial channel profiles without leaking held-out subjects into training.

These are separate evidence families. No single distance or classifier accuracy can establish all of them.